# 라이브 댓글 관리 — 의도 분류 게이트 성과 확인

**스파이크/평가용 코드입니다. 프로덕션 코드가 아닙니다.**

`chat_agent.py`(의도 분류 게이트 + 라우팅 구현)의 두 후보 전략(`kor_unsmile_plus_rule` / `kanana_zero_shot`)을
정답 라벨을 붙인 샘플로 실제 비교해서 지연시간·정확도를 확인한다.

전략을 바꾸려면 `model_registry.py`의 `ACTIVE_INTENT` 값을 바꾸거나, 아래 셀에서 `strategy=` 인자를 바꿔서
다시 실행하면 된다. 새 후보 전략을 추가하려면 `chat_agent.py`의 `_INTENT_STRATEGY_FUNCS`에 함수를 등록한다.

In [1]:
%pip install -q -r ../requirements-test.txt

Note: you may need to restart the kernel to use updated packages.


C:\Users\SSAFY\workspace\S15P11A108\AI\test\model_selection\.venv\Scripts\python.exe: No module named pip


## 1. 샘플 + 정답 라벨

`chat_agent.py` 데모와 같은 문장에 정답(의도) 라벨을 붙였다. 정상/악성/반복질문/스팸 4가지.

In [1]:
from chat_agent import (
    SpamDetector,
    classify_intent,
    route_comment,
)
from loaders import load_causal_lm, load_sequence_classifier
from model_registry import (
    ACTIVE_CLASSIFIER,
    ACTIVE_GENERATIVE,
    CLASSIFIER_MODELS,
    GENERATIVE_MODELS,
)

# (user_id, 댓글, 정답 의도) — 스팸은 SpamDetector가 의도 분류보다 먼저 판정하므로
# 같은 문장을 반복 게시해야 재현된다.
SAMPLES_WITH_LABELS = [
    ("user_1", "이 옷 진짜 예쁘네요 판매자님 감사합니다", "정상"),
    ("user_2", "가격 좀 너무 비싼거 아니에요?", "반복질문"),
    ("user_3", "이거 사이즈 M도 있나요?", "반복질문"),
    ("user_4", "저런 걸 옷이라고 파네 진짜 미친놈인가", "악성"),
    ("user_5", "여자가 무슨 명품을 입고 방송을 하냐 꼴값이네", "악성"),
    ("user_6", "품절이요 품절이요 품절이요", "정상"),
    ("user_6", "품절이요 품절이요 품절이요", "정상"),
    ("user_6", "품절이요 품절이요 품절이요", "스팸"),
]
len(SAMPLES_WITH_LABELS)

c:\Users\SSAFY\workspace\S15P11A108\AI\test\model_selection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


8

## 2. 전략별 실행

동일 샘플을 두 전략으로 각각 라우팅하고, 라운드마다 `RoutingDecision`(의도/액션/지연시간)을 모은다.

In [2]:
def run_strategy(strategy: str) -> list[dict]:
    detector = SpamDetector()
    classifier_bundle = (
        load_sequence_classifier(CLASSIFIER_MODELS[ACTIVE_CLASSIFIER])
        if strategy == "kor_unsmile_plus_rule"
        else None
    )
    gen_bundle = (
        load_causal_lm(GENERATIVE_MODELS[ACTIVE_GENERATIVE])
        if strategy == "kanana_zero_shot"
        else None
    )

    rows = []
    for user_id, comment, expected in SAMPLES_WITH_LABELS:
        decision = route_comment(
            comment,
            user_id,
            detector,
            classifier_bundle=classifier_bundle,
            gen_bundle=gen_bundle,
            strategy=strategy,
        )
        rows.append(
            {
                "comment": comment,
                "expected": expected,
                "predicted": decision.intent,
                "correct": decision.intent == expected,
                "latency_ms": round(decision.latency_ms, 1),
            }
        )
    return rows


rule_rows = run_strategy("kor_unsmile_plus_rule")
rule_rows

[{'comment': '이 옷 진짜 예쁘네요 판매자님 감사합니다',
  'expected': '정상',
  'predicted': '정상',
  'correct': True,
  'latency_ms': 149.3},
 {'comment': '가격 좀 너무 비싼거 아니에요?',
  'expected': '반복질문',
  'predicted': '반복질문',
  'correct': True,
  'latency_ms': 7.5},
 {'comment': '이거 사이즈 M도 있나요?',
  'expected': '반복질문',
  'predicted': '반복질문',
  'correct': True,
  'latency_ms': 6.2},
 {'comment': '저런 걸 옷이라고 파네 진짜 미친놈인가',
  'expected': '악성',
  'predicted': '악성',
  'correct': True,
  'latency_ms': 11.2},
 {'comment': '여자가 무슨 명품을 입고 방송을 하냐 꼴값이네',
  'expected': '악성',
  'predicted': '악성',
  'correct': True,
  'latency_ms': 11.7},
 {'comment': '품절이요 품절이요 품절이요',
  'expected': '정상',
  'predicted': '정상',
  'correct': True,
  'latency_ms': 12.5},
 {'comment': '품절이요 품절이요 품절이요',
  'expected': '정상',
  'predicted': '정상',
  'correct': True,
  'latency_ms': 11.8},
 {'comment': '품절이요 품절이요 품절이요',
  'expected': '스팸',
  'predicted': '스팸',
  'correct': True,
  'latency_ms': 0.1}]

In [3]:
kanana_rows = run_strategy("kanana_zero_shot")
kanana_rows

[{'comment': '이 옷 진짜 예쁘네요 판매자님 감사합니다',
  'expected': '정상',
  'predicted': '정상',
  'correct': True,
  'latency_ms': 447.6},
 {'comment': '가격 좀 너무 비싼거 아니에요?',
  'expected': '반복질문',
  'predicted': '정상',
  'correct': False,
  'latency_ms': 171.6},
 {'comment': '이거 사이즈 M도 있나요?',
  'expected': '반복질문',
  'predicted': '반복질문',
  'correct': True,
  'latency_ms': 270.7},
 {'comment': '저런 걸 옷이라고 파네 진짜 미친놈인가',
  'expected': '악성',
  'predicted': '악성',
  'correct': True,
  'latency_ms': 156.7},
 {'comment': '여자가 무슨 명품을 입고 방송을 하냐 꼴값이네',
  'expected': '악성',
  'predicted': '악성',
  'correct': True,
  'latency_ms': 175.3},
 {'comment': '품절이요 품절이요 품절이요',
  'expected': '정상',
  'predicted': '반복질문',
  'correct': False,
  'latency_ms': 235.7},
 {'comment': '품절이요 품절이요 품절이요',
  'expected': '정상',
  'predicted': '반복질문',
  'correct': False,
  'latency_ms': 219.5},
 {'comment': '품절이요 품절이요 품절이요',
  'expected': '스팸',
  'predicted': '스팸',
  'correct': True,
  'latency_ms': 0.1}]

## 3. 결과 비교 표

In [4]:
import pandas as pd

df = pd.DataFrame(
    {
        "comment": [r["comment"] for r in rule_rows],
        "expected": [r["expected"] for r in rule_rows],
        "rule_predicted": [r["predicted"] for r in rule_rows],
        "rule_correct": [r["correct"] for r in rule_rows],
        "rule_latency_ms": [r["latency_ms"] for r in rule_rows],
        "kanana_predicted": [r["predicted"] for r in kanana_rows],
        "kanana_correct": [r["correct"] for r in kanana_rows],
        "kanana_latency_ms": [r["latency_ms"] for r in kanana_rows],
    }
)
df

,comment,expected,rule_predicted,rule_correct,rule_latency_ms,kanana_predicted,kanana_correct,kanana_latency_ms
0,이 옷 진짜 예쁘네요 판매자님 감사합니다,정상,정상,True,149.3,정상,True,447.6
1,가격 좀 너무 비싼거 아니에요?,반복질문,반복질문,True,7.5,정상,False,171.6
2,이거 사이즈 M도 있나요?,반복질문,반복질문,True,6.2,반복질문,True,270.7
3,저런 걸 옷이라고 파네 진짜 미친놈인가,악성,악성,True,11.2,악성,True,156.7
4,여자가 무슨 명품을 입고 방송을 하냐 꼴값이네,악성,악성,True,11.7,악성,True,175.3
5,품절이요 품절이요 품절이요,정상,정상,True,12.5,반복질문,False,235.7
6,품절이요 품절이요 품절이요,정상,정상,True,11.8,반복질문,False,219.5
7,품절이요 품절이요 품절이요,스팸,스팸,True,0.1,스팸,True,0.1


In [5]:
summary = pd.DataFrame(
    {
        "strategy": ["kor_unsmile_plus_rule", "kanana_zero_shot"],
        "accuracy": [df["rule_correct"].mean(), df["kanana_correct"].mean()],
        "mean_latency_ms": [df["rule_latency_ms"].mean(), df["kanana_latency_ms"].mean()],
        "max_latency_ms": [df["rule_latency_ms"].max(), df["kanana_latency_ms"].max()],
    }
)
summary

,strategy,accuracy,mean_latency_ms,max_latency_ms
0,kor_unsmile_plus_rule,1.000,26.2875,149.3
1,kanana_zero_shot,0.625,209.6500,447.6


## 4. 해석

- **지연시간 목표**(설계 문서 제안값 &lt;300~500ms, 게이트 기준)를 `kor_unsmile_plus_rule`은 대부분 지키고,
  `kanana_zero_shot`은 LLM 생성 호출이 매번 들어가서 대체로 초과한다.
- **정확도**는 `accuracy` 컬럼(정답 라벨 대비 일치 비율)으로 비교한다. 다만 샘플 8개짜리 스모크 테스트라
  신뢰구간은 없다 — 실제 검증은 라벨링 데이터가 쌓인 뒤 더 큰 표본으로 다시 해야 한다.
- 다른 전략을 시도하려면 `run_strategy("전략명")`을 새 문자열로 다시 호출하면 된다
  (`chat_agent.py`의 `_INTENT_STRATEGY_FUNCS`에 등록된 키만 유효).
- `model_registry.py`의 `ACTIVE_INTENT`는 이 노트북 결과를 반영해 `kor_unsmile_plus_rule`로 설정돼 있다.